<a href="https://colab.research.google.com/github/shanileizer98-cyber/ml-insurance-project/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting Annual Claim Cost - Car Insurance
Machine Learning final assignment.
Goal: predict annual_claim_cost for a new driver, using only fields known before the contract is signed.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import norm

In [2]:
df = pd.read_csv('dataset_class.csv')
print(df.shape)
df.head()


(45000, 30)


,customer_id_hash,agent_code,country,area_type,region_density,winter_severity,driver_age,gender,marital_status,license_years,...,annual_km_estimated,pct_intercity,pct_night_driving,parking_type,current_premium,claim_count,at_fault,days_to_settlement,police_report_filed,annual_claim_cost
0,6db089eb,A043,ES,suburban,855,1.3,75,F,widowed,54,...,7500.0,5,15,lot,4450,1,0.0,27.0,0.0,6178
1,d3a860d7,A057,GB,suburban,1225,2.7,76,M,married,45,...,3000.0,25,10,street,1630,0,NaN,NaN,NaN,0
2,f0610253,A186,IL,suburban,1651,1.9,60,M,single,42,...,12500.0,0,5,garage,980,0,NaN,NaN,NaN,0
3,3137c022,A002,JP,urban,5441,7.7,55,M,divorced,34,...,10000.0,5,5,street,1810,0,NaN,NaN,NaN,0
4,a1513621,A028,FR,urban,3231,5.9,69,M,widowed,51,...,NaN,5,40,street,12290,0,NaN,NaN,NaN,0


In [3]:
df.isna().sum()

,0
customer_id_hash,0
agent_code,0
country,0
area_type,0
region_density,0
winter_severity,0
driver_age,0
gender,0
marital_status,0
license_years,0


In [4]:
y = df['annual_claim_cost']
print(y.describe())
print('share of zero:', (y == 0).mean())
print('share above 10,000:', (y > 10000).mean())


count     45000.000000
mean       3050.841644
std        7551.852703
min           0.000000
25%           0.000000
50%           0.000000
75%        3114.500000
max      224808.000000
Name: annual_claim_cost, dtype: float64
share of zero: 0.6490444444444444
share above 10,000: 0.09242222222222223


In [5]:
for col in ['gender', 'country', 'vehicle_type', 'usage_type', 'marital_status']:
    print(df[col].value_counts(), '\n')


gender
M         23624
F         20224
Male        232
m           195
female      187
male        187
f           177
Female      174
Name: count, dtype: int64 

country
US    9907
DE    5789
GB    5394
FR    4536
IT    4508
IL    4455
ES    3739
BR    3156
PL    2181
JP    1335
Name: count, dtype: int64 

vehicle_type
sedan         13916
small         12227
suv           10960
motorcycle     3072
pickup         2985
sports         1840
Name: count, dtype: int64 

usage_type
commute         18600
leisure         14265
admin_work       6226
delivery         3406
professional     2503
Name: count, dtype: int64 

marital_status
married     23290
single      13108
divorced     5550
widowed      3052
Name: count, dtype: int64 



## Data Overview

The dataset contains **45,000 drivers** and **30 columns**.

**Key findings:**
- The target variable (`annual_claim_cost`) is highly skewed: **65% of drivers cost the company nothing**, while a small minority cost tens of thousands of shekels.
- There are missing values in `prior_claims_5y` (3,142) and `annual_km_estimated` (4,311).
- The `gender` column is recorded in several different forms (`M`, `Male`, `m`, `male`, `F`, `Female`, `f`, `female`). These will be unified to `M`/`F` in `prepare_data`.

In [6]:
leak_cols = ['current_premium', 'claim_count', 'at_fault',
             'days_to_settlement', 'police_report_filed']
id_cols = ['customer_id_hash']


In [8]:
claim_cols = ['at_fault', 'days_to_settlement', 'police_report_filed']
df.groupby(df['annual_claim_cost'] > 0)[claim_cols].apply(lambda g: g.isna().mean())


,at_fault,days_to_settlement,police_report_filed
annual_claim_cost,,,
False,1.0,1.0,1.0
True,0.0,0.0,0.0


## Data Leakage (Section 2.2)

The model needs to predict the cost at the beginning of the year, when the driver is just signing the contract. So for every column I asked: does the insurance company know this on the day of signing? If not, I can't use it.

**Columns I removed:**
- `current_premium` - this is the premium paid at the end of the year, so we don't have it at the beginning.
- `claim_count` - the number of claims in this year, which is basically the answer.
- `at_fault`, `days_to_settlement` and `police_report_filed` - they are only filled if there was a claim.
- `customer_id_hash` - not leakage, just a random ID that doesn't say anything about the driver.

I also checked the three claim columns in the code: all of them are empty for every driver that cost 0, and full for every driver that cost more. This means that just the fact that the cell is empty already tells the model the answer.